# 🌿 Plant Disease Detection — Google Drive Training Pipeline

This notebook trains and benchmarks our core model architectures directly on **Google Drive** storage:
1. **MobileNetV3-Large** (Production Edge Model — 99.79% Accuracy)
2. **Pretrained ResNet-18** (Transfer Learning — 99.62% Accuracy)
3. **Deeper CNN + AdamW** (Custom CNN with tuned regularization — 99.09% Accuracy)

---
### ⚡ Step 1: Ensure GPU is Enabled
Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.

In [ ]:
# 1. Verify GPU is active
!nvidia-smi

### 📁 Step 2: Mount Google Drive & Set Working Directory

In [ ]:
import os
import sys
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Set project directory
PROJECT_PATH = '/content/drive/MyDrive/plant-disease-detection-dl'

# Ensure directory exists and change to it
if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    os.environ['PYTHONPATH'] = PROJECT_PATH
    if PROJECT_PATH not in sys.path:
        sys.path.insert(0, PROJECT_PATH)
    print(f"✓ Successfully set working directory to: {os.getcwd()}")
    print("Folder contents:")
    !ls -F
else:
    print(f"⚠️ Path '{PROJECT_PATH}' not found! Checking /content/drive/MyDrive:")
    !ls -F /content/drive/MyDrive

### 📦 Step 3: Install Required Dependencies

In [ ]:
!pip install -q datasets torchvision pandas matplotlib scikit-learn pillow python-multipart

### 📥 Step 4: Download Dataset & Create Splits (Run once)

In [ ]:
# If data/splits/train.csv doesn't exist, download and split
if not os.path.exists('data/splits/train.csv'):
    print("Downloading PlantVillage dataset & creating splits on Google Drive...")
    !python -m src.data.download_dataset
    !python -m src.data.create_splits
else:
    print("✓ Dataset and splits already exist on Google Drive!")

### 🚀 Step 5: Training Experiments Runner
Trains models and automatically saves checkpoints to `experiments/<model_name>/best_model.pt` on Google Drive.

In [ ]:
import time

experiments_to_run = [
    ("MobileNetV3 Edge Model", "src.experiments.mobilenet_v3"),
    ("ResNet-18 Transfer Learning", "src.experiments.resnet18"),
    ("Deeper CNN with AdamW", "src.experiments.deeper_cnn_adamw")
]

for title, module_path in experiments_to_run:
    print(f"\n{'=' * 70}")
    print(f"▶ Training: {title} ({module_path})")
    print(f"Started at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'=' * 70}\n")
    
    !python -m {module_path} --train
    print(f"\n✓ Finished {title}! Checkpoint saved to Google Drive.")

### 🏆 Step 6: Generate Production Benchmark & Leaderboard
Compares all trained models across **Accuracy, Model Size (MB), and Inference Latency (ms)**.

In [ ]:
!python -m src.training.benchmark